In [2]:
import os
import json
import pandas as pd
import pyarrow.dataset as ds

MODEL_DATA_DIR = "data/raw"

DATA_PATH = os.path.join(
    MODEL_DATA_DIR,
    "m5_model_features.parquet"
)

CONFIG_PATH = os.path.join(
    MODEL_DATA_DIR,
    "feature_config.json"
)

print("Data path:", DATA_PATH)
print("Config path:", CONFIG_PATH)

Data path: data/raw/m5_model_features.parquet
Config path: data/raw/feature_config.json


In [3]:
with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

TARGET = config["TARGET"]
FEATURES = config["FEATURES"]

CALENDAR_FEATURES = config["CALENDAR_FEATURES"]
PRICE_FEATURES = config["PRICE_FEATURES"]
LAG_FEATURES = config["LAG_FEATURES"]
ROLLING_FEATURES = config["ROLLING_FEATURES"]
DEMAND_FEATURES = config["DEMAND_FEATURES"]
EVENT_FEATURES = config["EVENT_FEATURES"]
SNAP_FEATURES = config["SNAP_FEATURES"]

ID_COLS = config["ID_COLS"]
META_COLS = config["META_COLS"]

HOLDOUT_DAYS = config["HOLDOUT_DAYS"]
FINAL_DAY = config["FINAL_DAY"]
CUTOFF_DAY = config["CUTOFF_DAY"]

print("Target:", TARGET)
print("Number of features:", len(FEATURES))
print("Cutoff day:", CUTOFF_DAY)
print("Final day:", FINAL_DAY)

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/feature_config.json'

In [4]:
dset = ds.dataset(DATA_PATH, format="parquet")

print("Dataset loaded")
print("Number of columns:", len(dset.schema.names))
print(dset.schema.names)

FileNotFoundError: data/raw/m5_model_features.parquet

In [5]:
print("Columns:", len(dset.schema.names))
print("Rows:", dset.count_rows())

print("\nColumns:")
print(dset.schema.names)

print("\nExpected model features:", len(FEATURES))
print("Target:", TARGET)
print("Cutoff:", CUTOFF_DAY)
print("Final day:", FINAL_DAY)

NameError: name 'dset' is not defined

In [6]:
TRAIN_COLS = FEATURES + [TARGET]

train_days = [f"d_{i}" for i in range(1, CUTOFF_DAY + 1)]

train_dset = dset.to_table(
    columns=TRAIN_COLS,
    filter=ds.field("d").isin(train_days)
)

print(train_dset.num_rows)
print(train_dset.num_columns)

NameError: name 'FEATURES' is not defined

In [7]:
import pyarrow.parquet as pq

TRAIN_DATA_PATH = "data/raw/train_features.parquet"

pq.write_table(
    train_dset,
    TRAIN_DATA_PATH,
    compression="snappy"
)

print("Saved:", TRAIN_DATA_PATH)

NameError: name 'train_dset' is not defined

In [ ]:
train_dset = ds.dataset(TRAIN_DATA_PATH, format="parquet")

print("Training rows:", train_dset.count_rows())
print("Training columns:", len(train_dset.schema.names))

In [ ]:
VALIDATION_DAYS = 28
VALIDATION_END = CUTOFF_DAY
VALIDATION_START = CUTOFF_DAY - VALIDATION_DAYS + 1

print(
    "Validation:",
    f"d_{VALIDATION_START}",
    "→",
    f"d_{VALIDATION_END}"
)

In [ ]:
VAL_COLS = ["id", "d", "sales", "lag_1", "lag_7", "lag_28"]

val_table = dset.to_table(
    columns=VAL_COLS,
    filter=ds.field("d").isin(
        [f"d_{i}" for i in range(VALIDATION_START, VALIDATION_END + 1)]
    )
)

val_df = val_table.to_pandas()

print("Validation shape:", val_df.shape)
print(val_df.head())

In [ ]:
# Baseline columns

val_df["pred_naive"] = val_df["lag_1"]
val_df["pred_seasonal_7"] = val_df["lag_7"]
val_df["pred_seasonal_28"] = val_df["lag_28"]

print(val_df[
    [
        "sales",
        "pred_naive",
        "pred_seasonal_7",
        "pred_seasonal_28"
    ]
].head())

basline performance calculate

In [ ]:
from sklearn.metrics import mean_absolute_error

for name in ["pred_naive", "pred_seasonal_7", "pred_seasonal_28"]:
    mae = mean_absolute_error(val_df["sales"], val_df[name])
    print(name, "MAE:", round(mae, 4))

7 days aur 28 day moving average baseline

In [ ]:
VAL_COLS = ["id", "d", "sales", "roll_mean_7", "roll_mean_28"]

val_table = dset.to_table(
    columns=VAL_COLS,
    filter=ds.field("d").isin(
        [f"d_{i}" for i in range(VALIDATION_START, VALIDATION_END + 1)]
    )
)

val_df = val_table.to_pandas()

val_df["pred_ma_7"] = val_df["roll_mean_7"]
val_df["pred_ma_28"] = val_df["roll_mean_28"]

print(val_df[["sales", "pred_ma_7", "pred_ma_28"]].head())

MAE calculate for these

In [ ]:
for name in ["pred_ma_7", "pred_ma_28"]:
    mae = mean_absolute_error(val_df["sales"], val_df[name])
    print(name, "MAE:", round(mae, 4))

lightgbm

In [ ]:
print("Features:", len(FEATURES))
print("Target:", TARGET)
print("Training rows:", train_dset.count_rows())

In [ ]:
import lightgbm as lgb
print("LightGBM version:",lgb._version_)

In [ ]:
import lightgbm as lgb
import pyarrow.dataset as ds
import pandas as pd
import numpy as np

MODEL_START = 1913 - 365 + 1

model_cols = FEATURES + [TARGET]

days = [f"d_{i}" for i in range(MODEL_START, 1914)]

table = dset.to_table(
    columns=model_cols,
    filter=ds.field("d").isin(days)
)

df_lgb = table.to_pandas()

X = df_lgb[FEATURES]
y = df_lgb[TARGET]

print("Training shape:", X.shape)

model = lgb.LGBMRegressor(
    objective="poisson",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X,
    y
)

print("LightGBM training complete")

MAE for these 365 days only

In [ ]:

VAL_MODEL_COLS = FEATURES + [TARGET]

val_table = dset.to_table(
    columns=VAL_MODEL_COLS,
    filter=ds.field("d").isin(
        [f"d_{i}" for i in range(VALIDATION_START, VALIDATION_END + 1)]
    )
)

val_model_df = val_table.to_pandas()

val_X = val_model_df[FEATURES]
val_y = val_model_df[TARGET]

val_pred = model.predict(val_X)

mae = mean_absolute_error(val_y, val_pred)

print("Validation shape:", val_X.shape)
print("LightGBM MAE:", round(mae, 4))

Final model

In [ ]:
import dask
import dask.dataframe as dd
import lightgbm as lgb

print("Dask:", dask.__version__)
print("LightGBM:", lgb.__version__)

train_dd = dd.read_parquet(
    TRAIN_DATA_PATH,
    columns=FEATURES + [TARGET]
)

print("Partitions:", train_dd.npartitions)
print("Columns:", len(train_dd.columns))

In [ ]:
from dask.distributed import Client
from lightgbm.dask import DaskLGBMRegressor

client = Client(
    n_workers=4,
    threads_per_worker=1
)

train_dd = dd.read_parquet(
    TRAIN_DATA_PATH,
    columns=FEATURES + [TARGET]
)

train_dd = train_dd.repartition(npartitions=16)

X_dd = train_dd[FEATURES]
y_dd = train_dd[TARGET]

print(client)
print("Partitions:", train_dd.npartitions)
print("Features:", len(FEATURES))

In [ ]:
from importlib.metadata import version

print("LightGBM:", version("lightgbm"))
print("Dask:", version("dask"))
print("Distributed:", version("distributed"))

In [ ]:
import lightgbm.dask as lgb_dask
import inspect

print(inspect.signature(lgb_dask._train))

In [ ]:
import gc
import lightgbm as lgb
import pyarrow.dataset as ds

# Free Dask workers/memory
try:
    client.close()
except:
    pass

del train_dd, X_dd, y_dd
gc.collect()

# Clean validation training period
TRAIN_END = VALIDATION_START - 1
TRAIN_START = TRAIN_END - 365 + 1

TRAIN_DAYS = [f"d_{i}" for i in range(TRAIN_START, TRAIN_END + 1)]

train_table = dset.to_table(
    columns=FEATURES + [TARGET],
    filter=ds.field("d").isin(TRAIN_DAYS)
)

train_df = train_table.to_pandas()

print("Training days:", TRAIN_START, "to", TRAIN_END)
print("Training shape:", train_df.shape)

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

lgb_model = lgb.LGBMRegressor(
    objective="poisson",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(X_train, y_train)

print("LightGBM training complete")

In [ ]:
from sklearn.metrics import mean_absolute_error
import pyarrow.dataset as ds

VAL_MODEL_COLS = FEATURES + [TARGET]

val_table = dset.to_table(
    columns=VAL_MODEL_COLS,
    filter=ds.field("d").isin(
        [f"d_{i}" for i in range(VALIDATION_START, VALIDATION_END + 1)]
    )
)

val_model_df = val_table.to_pandas()

val_X = val_model_df[FEATURES]
val_y = val_model_df[TARGET]

val_pred = lgb_model.predict(val_X)

val_mae = mean_absolute_error(val_y, val_pred)

print(
    "Validation period:",
    f"d_{VALIDATION_START} -> d_{VALIDATION_END}"
)
print("LightGBM Validation MAE:", round(val_mae, 4))

Two stage model

In [ ]:
import lightgbm as lgb

# Stage 1: Will a sale happen?
y_occurrence = (y_train > 0).astype("int8")

occurrence_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

occurrence_model.fit(X_train, y_occurrence)

print("Stage 1 occurrence model trained")

In [ ]:
# Stage 2: How many units are sold if a sale happens?
positive_mask = y_train > 0

X_size = X_train.loc[positive_mask]
y_size = y_train.loc[positive_mask]

size_model = lgb.LGBMRegressor(
    objective="poisson",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

size_model.fit(X_size, y_size)

print("Stage 2 size model trained")
print("Positive-sale training rows:", len(y_size))

In [ ]:
# Validation predictions
val_prob = occurrence_model.predict_proba(val_X)[:, 1]
val_size = size_model.predict(val_X)

# Two-stage prediction
two_stage_pred = val_prob * val_size

# MAE
two_stage_mae = mean_absolute_error(val_y, two_stage_pred)

print("Two-Stage Validation MAE:", round(two_stage_mae, 4))

XGBOOST

In [ ]:
import xgboost as xgb
print("XGBoost:",xgb._version_)

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    objective="count:poisson",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

print("XGBoost training complete")

xgboost trainning complete

In [ ]:
xgb_val_pred = xgb_model.predict(val_X)

xgb_mae = mean_absolute_error(val_y, xgb_val_pred)

print("XGBoost Validation MAE:", round(xgb_mae, 4))

In [ ]:
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": lgb_model.feature_importances_
}).sort_values("importance", ascending=False)

print(feature_importance)

In [ ]:
print(feature_importance.head(10))

In [ ]:
import sys
print("Notebook Python",sys.executable)
!{sys.executable}-m pip install shap

In [ ]:
import shap  
print("SHAP:",shap._version_)

In [ ]:
shap_sample = val_model_df[FEATURES].sample(
    n=min(5000, len(val_model_df)),
    random_state=42
)

explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(shap_sample)

print("SHAP analysis ready")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

shap_importance = pd.DataFrame({
    "feature": shap_sample.columns,
    "importance": np.abs(shap_values).mean(axis=0)
}).sort_values("importance", ascending=True).tail(15)

plt.figure(figsize=(10, 7))

bars = plt.barh(
    shap_importance["feature"],
    shap_importance["importance"]
)

plt.xlabel("Mean |SHAP value|")
plt.ylabel("Feature")
plt.title(
    "Top 15 Features - SHAP Feature Importance",
    fontsize=15,
    pad=15
)

for bar, value in zip(bars, shap_importance["importance"]):
    plt.text(
        bar.get_width(),
        bar.get_y() + bar.get_height()/2,
        f"{value:.3f}",
        va="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from sklearn.metrics import mean_absolute_error

VAL_DAYS = list(range(1886, 1914))

# Validation data
val_table = dset.to_table(
    columns=["id", "d", "sales"] + FEATURES,
    filter=ds.field("d").isin([f"d_{i}" for i in VAL_DAYS])
)
val_rec = val_table.to_pandas()

# History before validation
HIST_DAYS = list(range(1830, 1886))

hist_table = dset.to_table(
    columns=["id", "d", "sales"],
    filter=ds.field("d").isin([f"d_{i}" for i in HIST_DAYS])
)
hist = hist_table.to_pandas()

history = hist.pivot(
    index="id",
    columns="d",
    values="sales"
)

all_preds = []
all_actual = []

DYNAMIC_FEATURES = [
    "lag_1", "lag_7", "lag_14", "lag_28", "lag_56",
    "roll_mean_7", "roll_std_7",
    "roll_mean_28", "roll_std_28",
    "zero_ratio_7", "zero_ratio_28",
    "days_since_last_sale"
]

for day in VAL_DAYS:
    day_str = f"d_{day}"
    day_df = val_rec[val_rec["d"] == day_str].copy()

    ids = day_df["id"].values

    recent_cols = [
        f"d_{i}" for i in range(day - 56, day)
    ]

    recent = history.reindex(
        ids, columns=recent_cols
    ).fillna(0).values

    day_df["lag_1"] = recent[:, -1]
    day_df["lag_7"] = recent[:, -7]
    day_df["lag_14"] = recent[:, -14]
    day_df["lag_28"] = recent[:, -28]
    day_df["lag_56"] = recent[:, 0]

    day_df["roll_mean_7"] = recent[:, -7:].mean(axis=1)
    day_df["roll_std_7"] = recent[:, -7:].std(axis=1)

    day_df["roll_mean_28"] = recent[:, -28:].mean(axis=1)
    day_df["roll_std_28"] = recent[:, -28:].std(axis=1)

    day_df["zero_ratio_7"] = (
        recent[:, -7:] == 0
    ).mean(axis=1)

    day_df["zero_ratio_28"] = (
        recent[:, -28:] == 0
    ).mean(axis=1)

    days_since = []

    for row in recent:
        nonzero = np.where(row > 0)[0]
        days_since.append(
            56 if len(nonzero) == 0
            else 55 - nonzero[-1]
        )

    day_df["days_since_last_sale"] = days_since

    X_day = day_df[FEATURES]

    # Stage 1: Probability of a sale
    prob = occurrence_model.predict_proba(X_day)[:, 1]

    # Stage 2: Expected units sold
    size = size_model.predict(X_day)

    # Final prediction
    day_pred = prob * size

    all_preds.extend(day_pred)
    all_actual.extend(day_df["sales"].values)

# IMPORTANT: use predictions as history for next day
history[day_str] = pd.Series(day_pred, index=ids)

recursive_mae = mean_absolute_error(all_actual, all_preds)

print("Recursive validation:", "d_1886 → d_1913")
print("Recursive Two-Stage MAE:", round(recursive_mae, 4))

recursive lightgbm validation

In [ ]:
# Recursive LightGBM validation

history_lgb = hist.pivot(index="id", columns="d", values="sales")

lgb_preds = []
lgb_actual = []

for day in VAL_DAYS:
    day_str = f"d_{day}"
    day_df = val_rec[val_rec["d"] == day_str].copy()

    ids = day_df["id"].values

    recent_cols = [f"d_{i}" for i in range(day - 56, day)]
    recent = history_lgb.reindex(
        ids, columns=recent_cols
    ).fillna(0).values

    day_df["lag_1"] = recent[:, -1]
    day_df["lag_7"] = recent[:, -7]
    day_df["lag_14"] = recent[:, -14]
    day_df["lag_28"] = recent[:, -28]
    day_df["lag_56"] = recent[:, 0]

    day_df["roll_mean_7"] = recent[:, -7:].mean(axis=1)
    day_df["roll_std_7"] = recent[:, -7:].std(axis=1)
    day_df["roll_mean_28"] = recent[:, -28:].mean(axis=1)
    day_df["roll_std_28"] = recent[:, -28:].std(axis=1)

    day_df["zero_ratio_7"] = (recent[:, -7:] == 0).mean(axis=1)
    day_df["zero_ratio_28"] = (recent[:, -28:] == 0).mean(axis=1)

    days_since = []
    for row in recent:
        nonzero = np.where(row > 0)[0]
        days_since.append(
            56 if len(nonzero) == 0 else 55 - nonzero[-1]
        )

    day_df["days_since_last_sale"] = days_since

    X_day = day_df[FEATURES]

    day_pred = lgb_model.predict(X_day)

    lgb_preds.extend(day_pred)
    lgb_actual.extend(day_df["sales"].values)

    # Prediction becomes history for next day
    history_lgb[day_str] = pd.Series(day_pred, index=ids)

lgb_recursive_mae = mean_absolute_error(lgb_actual, lgb_preds)

print("Recursive validation: d_1886 → d_1913")
print("Recursive LightGBM MAE:", round(lgb_recursive_mae, 4))

In [ ]:
# Recursive XGBoost validation

history_xgb = hist.pivot(index="id", columns="d", values="sales")

xgb_preds = []
xgb_actual = []

for day in VAL_DAYS:
    day_str = f"d_{day}"
    day_df = val_rec[val_rec["d"] == day_str].copy()

    ids = day_df["id"].values

    recent_cols = [f"d_{i}" for i in range(day - 56, day)]
    recent = history_xgb.reindex(
        ids, columns=recent_cols
    ).fillna(0).values

    day_df["lag_1"] = recent[:, -1]
    day_df["lag_7"] = recent[:, -7]
    day_df["lag_14"] = recent[:, -14]
    day_df["lag_28"] = recent[:, -28]
    day_df["lag_56"] = recent[:, 0]

    day_df["roll_mean_7"] = recent[:, -7:].mean(axis=1)
    day_df["roll_std_7"] = recent[:, -7:].std(axis=1)
    day_df["roll_mean_28"] = recent[:, -28:].mean(axis=1)
    day_df["roll_std_28"] = recent[:, -28:].std(axis=1)

    day_df["zero_ratio_7"] = (recent[:, -7:] == 0).mean(axis=1)
    day_df["zero_ratio_28"] = (recent[:, -28:] == 0).mean(axis=1)

    days_since = []

    for row in recent:
        nonzero = np.where(row > 0)[0]
        days_since.append(
            56 if len(nonzero) == 0 else 55 - nonzero[-1]
        )

    day_df["days_since_last_sale"] = days_since

    X_day = day_df[FEATURES]

    day_pred = xgb_model.predict(X_day)

    xgb_preds.extend(day_pred)
    xgb_actual.extend(day_df["sales"].values)

    # Prediction becomes history for next day
    history_xgb[day_str] = pd.Series(day_pred, index=ids)

xgb_recursive_mae = mean_absolute_error(xgb_actual, xgb_preds)

print("Recursive validation: d_1886 → d_1913")
print("Recursive XGBoost MAE:", round(xgb_recursive_mae, 4))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

comparison = pd.DataFrame({
    "Model": ["LightGBM", "Two-Stage", "XGBoost"],
    "Normal MAE": [0.9741, 0.9691, 0.9706],
    "Recursive MAE": [1.1669, 1.1453, 1.1113]
})

ax = comparison.set_index("Model").plot(
    kind="bar",
    figsize=(10, 6),
    width=0.75
)

plt.title("Model Comparison: Normal vs Recursive Validation", fontsize=15)
plt.xlabel("Model")
plt.ylabel("MAE (Lower is Better)")
plt.xticks(rotation=0)
plt.legend(title="Validation Type")
plt.grid(axis="y", alpha=0.25)

for container in ax.containers:
    ax.bar_label(container, fmt="%.4f", padding=3)

plt.tight_layout()
plt.show()

SARIMAX  

In [ ]:
# ============================================================
# SARIMAX SAMPLE EXPERIMENT + RECURSIVE XGBOOST COMPARISON
# Reconstructed from the previous image and new screenshots
# ============================================================

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. Select 30 random time series
# ------------------------------------------------------------

sample_ids = (
    val_rec["id"]
    .drop_duplicates()
    .sample(n=30, random_state=42)
    .tolist()
)

train_days = [f"d_{i}" for i in range(1521, 1886)]
all_days = [f"d_{i}" for i in range(1521, 1914)]

# ------------------------------------------------------------
# 2. Load only the selected series
# ------------------------------------------------------------

sample_table = ds.to_table(
    columns=["id", "d", "sales"],
    filter=(
        ds.field("id").isin(sample_ids)
        & ds.field("d").isin(all_days)
    )
)

sample_data = sample_table.to_pandas()

sample_wide = (
    sample_data
    .pivot(index="id", columns="d", values="sales")
    .reindex(index=sample_ids, columns=all_days)
    .fillna(0)
)

# ------------------------------------------------------------
# 3. Train SARIMAX and forecast 28 days
# ------------------------------------------------------------

sarimax_predictions = {}
sarimax_actual = {}

for sid in sample_ids:

    y_train_series = (
        sample_wide.loc[sid, train_days]
        .values.astype(float)
    )

    y_test_series = (
        sample_wide.loc[
            sid,
            [f"d_{i}" for i in range(1886, 1914)]
        ]
        .values.astype(float)
    )

    try:
        model = SARIMAX(
            y_train_series,
            order=(1, 0, 0),
            seasonal_order=(1, 0, 0, 7),
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        fitted = model.fit(disp=False, maxiter=50)

        forecast = np.maximum(
            fitted.forecast(steps=28), 0
        )

        sarimax_predictions[sid] = forecast
        sarimax_actual[sid] = y_test_series

    except Exception:
        print("Skipped series:", sid)

print(
    "SARIMAX completed for",
    len(sarimax_predictions),
    "series"
)

# ------------------------------------------------------------
# 4. Calculate SARIMAX MAE
# ------------------------------------------------------------

sarimax_pred_flat = np.concatenate(
    list(sarimax_predictions.values())
)

sarimax_actual_flat = np.concatenate(
    list(sarimax_actual.values())
)

sarimax_mae_sample = mean_absolute_error(
    sarimax_actual_flat,
    sarimax_pred_flat
)

# ------------------------------------------------------------
# 5. Recursive XGBoost on the same successful series
# ------------------------------------------------------------
# Prerequisites from earlier notebook cells:
# val_rec      : validation data containing id, d, sales
# xgb_model    : already-trained XGBoost model
# FEATURES     : list of feature column names
# history data : sales history for the selected series

successful_ids = list(sarimax_predictions.keys())

val_sample = val_rec[
    val_rec["id"].isin(successful_ids)
].copy()

history_xgb_sample = sample_wide.loc[
    successful_ids, train_days
].copy()

xgb_sample_preds = []
xgb_sample_actual = []

for day in range(1886, 1914):

    day_str = f"d_{day}"

    day_df = val_sample[
        val_sample["d"] == day_str
    ].copy()

    ids = day_df["id"].values

    recent_cols = [
        f"d_{i}" for i in range(day - 56, day)
    ]

    recent = (
        history_xgb_sample
        .reindex(index=ids, columns=recent_cols)
        .fillna(0)
        .values
    )

    # Dynamic time-series features
    dynamic_features = {
        "lag_1": recent[:, -1],
        "lag_7": recent[:, -7],
        "lag_14": recent[:, -14],
        "lag_28": recent[:, -28],
        "lag_56": recent[:, 0],

        "roll_mean_7": recent[:, -7:].mean(axis=1),
        "roll_std_7": recent[:, -7:].std(axis=1),

        "roll_mean_28": recent[:, -28:].mean(axis=1),
        "roll_std_28": recent[:, -28:].std(axis=1),

        "zero_ratio_7": (
            recent[:, -7:] == 0
        ).mean(axis=1),

        "zero_ratio_28": (
            recent[:, -28:] == 0
        ).mean(axis=1),

        "days_since_last_sale": np.array([
            56 if not np.any(row > 0)
            else 56 - np.where(row > 0)[0][-1]
            for row in recent
        ])
    }

    # Add generated features to the daily data
    for col, values in dynamic_features.items():
        if col in day_df.columns:
            day_df[col] = values

    # Predict the current day
    day_pred = np.maximum(
        xgb_model.predict(day_df[FEATURES]),
        0
    )

    xgb_sample_preds.extend(day_pred)
    xgb_sample_actual.extend(
        day_df["sales"].values
    )

    # Feed predictions into history for the next day
    history_xgb_sample[day_str] = pd.Series(
        day_pred,
        index=ids
    )

# ------------------------------------------------------------
# 6. Compare models using MAE
# ------------------------------------------------------------

xgb_mae_sample = mean_absolute_error(
    xgb_sample_actual,
    xgb_sample_preds
)

comparison_sample = pd.DataFrame({
    "Model": ["SARIMAX", "XGBoost"],
    "Recursive MAE": [
        sarimax_mae_sample,
        xgb_mae_sample
    ]
}).sort_values("Recursive MAE")

print("\nSample comparison: d_1886 -> d_1913")
print(
    comparison_sample.to_string(index=False)
)